# CSL 422 – Machine Learning Lab
## Assignment 7 — Artificial Neural Network
**Dataset:** Pima Indians Diabetes Data  
**Date:** 21 April, 2026

---

### Objective
Develop and evaluate an Artificial Neural Network (ANN) using the Pima Indians Diabetes dataset.

**Tasks:**
1. Load dataset. Build ANN with ≥2 hidden layers, ReLU (hidden) and Sigmoid (output).
2. Compile model, define epochs & batch size, train.
3. Evaluate performance by varying epochs and batch sizes.
4. Try different activation functions and compare accuracy.

## Step 1 — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print("All libraries imported successfully!")

## Step 2 — Load and Explore the Dataset

In [ ]:
# Load Pima Indians Diabetes Dataset
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
           'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

df = pd.read_csv(url, names=columns)
print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")
df.head(10)

In [ ]:
# Basic statistics
print("=== Dataset Info ===")
df.info()
print("\n=== Statistical Summary ===")
df.describe()

In [ ]:
# Class distribution
print("Class Distribution:")
print(df['Outcome'].value_counts())
print(f"\nDiabetic:     {df['Outcome'].sum()} ({df['Outcome'].mean()*100:.1f}%)")
print(f"Non-Diabetic: {(df['Outcome'] == 0).sum()} ({(1 - df['Outcome'].mean())*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution pie
axes[0].pie(df['Outcome'].value_counts(), labels=['Non-Diabetic', 'Diabetic'],
            autopct='%1.1f%%', colors=['#4CAF50', '#F44336'], startangle=90)
axes[0].set_title('Class Distribution', fontsize=13)

# Feature correlations
corr = df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1], linewidths=0.5)
axes[1].set_title('Feature Correlation Heatmap', fontsize=13)

plt.tight_layout()
plt.show()

## Step 3 — Preprocessing

In [ ]:
# Replace 0s in biological columns with NaN, then fill with median
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[zero_cols] = df[zero_cols].replace(0, np.nan)
df.fillna(df.median(), inplace=True)

print("Missing values after cleaning:", df.isnull().sum().sum())

# Split features and target
X = df.drop('Outcome', axis=1).values
y = df['Outcome'].values

# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")
print(f"Features:         {X_train.shape[1]}")

## Step 4 — Build the ANN Model

> Architecture: **Input → Dense(64, ReLU) → Dropout → Dense(32, ReLU) → Dropout → Dense(1, Sigmoid)**

In [ ]:
def build_ann(activation='relu', input_dim=8):
    """Build ANN with at least 2 hidden layers."""
    model = Sequential([
        Dense(64, activation=activation, input_dim=input_dim),
        Dropout(0.3),
        Dense(32, activation=activation),
        Dropout(0.2),
        Dense(1, activation='sigmoid')   # Output layer: Sigmoid for binary classification
    ])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# Build and summarize base model
model = build_ann(activation='relu')
model.summary()

## Step 5 — Train the Model (Base: epochs=100, batch_size=32)

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

print(f"\nTraining stopped at epoch: {len(history.history['loss'])}")

In [ ]:
def plot_history(history, title='Model Training History'):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    
    axes[0].plot(history.history['accuracy'], label='Train Accuracy', color='royalblue')
    axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', color='orange')
    axes[0].set_title(f'{title} — Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history.history['loss'], label='Train Loss', color='crimson')
    axes[1].plot(history.history['val_loss'], label='Val Loss', color='darkorange')
    axes[1].set_title(f'{title} — Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_history(history, title='Base Model (ReLU, epochs=100, batch=32)')

## Step 6 — Evaluate Base Model

In [ ]:
def evaluate_model(model, X_test, y_test, label='Model'):
    y_pred_prob = model.predict(X_test)
    y_pred = (y_pred_prob > 0.5).astype(int).flatten()
    acc = accuracy_score(y_test, y_pred)

    print(f"\n{'='*40}")
    print(f"  {label}")
    print(f"{'='*40}")
    print(f"Test Accuracy: {acc*100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Non-Diabetic', 'Diabetic']))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Non-Diabetic', 'Diabetic'],
                yticklabels=['Non-Diabetic', 'Diabetic'])
    plt.title(f'Confusion Matrix — {label}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.show()
    return acc

base_acc = evaluate_model(model, X_test, y_test, label='Base Model (ReLU, epochs=100, batch=32)')

## Task 3 — Varying Epochs and Batch Sizes

In [ ]:
configs = [
    {'epochs': 50,  'batch_size': 16},
    {'epochs': 50,  'batch_size': 32},
    {'epochs': 100, 'batch_size': 32},
    {'epochs': 100, 'batch_size': 64},
    {'epochs': 150, 'batch_size': 32},
    {'epochs': 200, 'batch_size': 64},
]

results_eb = []

for cfg in configs:
    m = build_ann(activation='relu')
    h = m.fit(
        X_train, y_train,
        epochs=cfg['epochs'],
        batch_size=cfg['batch_size'],
        validation_split=0.2,
        verbose=0
    )
    y_pred = (m.predict(X_test) > 0.5).astype(int).flatten()
    acc = accuracy_score(y_test, y_pred) * 100
    results_eb.append({
        'Epochs': cfg['epochs'],
        'Batch Size': cfg['batch_size'],
        'Accuracy (%)': round(acc, 2)
    })
    print(f"  epochs={cfg['epochs']:3d}, batch={cfg['batch_size']:2d} → Accuracy: {acc:.2f}%")

df_eb = pd.DataFrame(results_eb)
print("\n")
display(df_eb)

In [ ]:
# Visualize Epochs vs Batch Size effect
fig, ax = plt.subplots(figsize=(10, 5))

for batch in df_eb['Batch Size'].unique():
    subset = df_eb[df_eb['Batch Size'] == batch]
    ax.plot(subset['Epochs'], subset['Accuracy (%)'], marker='o', label=f'Batch Size={batch}')

ax.set_title('Accuracy vs Epochs for Different Batch Sizes', fontsize=13)
ax.set_xlabel('Epochs')
ax.set_ylabel('Accuracy (%)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Task 4 — Comparing Activation Functions

In [ ]:
activations = ['relu', 'tanh', 'sigmoid', 'elu', 'selu']
results_act = []

for act in activations:
    m = build_ann(activation=act)
    m.fit(
        X_train, y_train,
        epochs=100,
        batch_size=32,
        validation_split=0.2,
        verbose=0
    )
    y_pred = (m.predict(X_test) > 0.5).astype(int).flatten()
    acc = accuracy_score(y_test, y_pred) * 100
    results_act.append({'Activation': act, 'Accuracy (%)': round(acc, 2)})
    print(f"  Activation: {act:8s} → Accuracy: {acc:.2f}%")

df_act = pd.DataFrame(results_act).sort_values('Accuracy (%)', ascending=False)
print("\n")
display(df_act)

In [ ]:
colors = ['#4CAF50' if acc == df_act['Accuracy (%)'].max() else '#2196F3'
          for acc in df_act['Accuracy (%)']]

plt.figure(figsize=(9, 5))
bars = plt.bar(df_act['Activation'], df_act['Accuracy (%)'], color=colors, edgecolor='black', linewidth=0.6)

for bar, val in zip(bars, df_act['Accuracy (%)']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.2f}%', ha='center', va='bottom', fontsize=10)

plt.title('Accuracy Comparison Across Activation Functions', fontsize=13)
plt.xlabel('Activation Function')
plt.ylabel('Test Accuracy (%)')
plt.ylim(df_act['Accuracy (%)'].min() - 5, 100)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Summary & Observations

In [ ]:
best_act = df_act.iloc[0]
best_eb  = df_eb.loc[df_eb['Accuracy (%)'].idxmax()]

print("=" * 55)
print("          EXPERIMENT SUMMARY")
print("=" * 55)
print(f"\n[Task 1-2] Base Model (ReLU, epochs=100, batch=32)")
print(f"           Accuracy: {base_acc*100:.2f}%")

print(f"\n[Task 3]   Best Epochs/Batch Config:")
print(f"           Epochs={int(best_eb['Epochs'])}, Batch={int(best_eb['Batch Size'])} → {best_eb['Accuracy (%)']}%")

print(f"\n[Task 4]   Best Activation Function:")
print(f"           '{best_act['Activation']}' → {best_act['Accuracy (%)']}%")

print("\n" + "=" * 55)
print("\nKey Observations:")
print("  • ReLU and ELU generally outperform Sigmoid in hidden layers")
print("  • Smaller batch sizes can improve generalization but train slower")
print("  • Early stopping helps prevent overfitting")
print("  • StandardScaler is critical for ANN convergence")